In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 8.12 Berry Phase, Wannier Functions, and the SSH Model

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VIII — Electronic Structure and Many-Body Matter",
    number="8.12",
    title="Berry Phase, Wannier Functions, and the SSH Model",
    blurb="Movement III closes with the discovery that Bloch states carry "
    "geometry. A discrete Berry phase, built in six lines and certified on "
    "Berry's own spin-1/2 example, is turned on the dimerized chain of "
    "§8.9 — the Su–Schrieffer–Heeger model — where it "
    "quantizes to 0 or π, predicts which chains carry zero-energy edge "
    "states (they appear at 10⁻¹⁰, on one sublattice, exactly "
    "when promised), builds exponentially localized Wannier functions whose "
    "centers are the modern theory of polarization, and jumps by exactly "
    "e/2 at the transition. A chain with winding number two, and four edge "
    "states, closes the case.",
    difficulty="advanced",
    estimate="150–180 min",
)

## Notebook overview

[§8.9](tight-binding.ipynb) ended with a provocation: the half-filled chain
lowers its energy by dimerizing, and a dimerized chain can dimerize *two
ways* — strong bond first or weak bond first. The two patterns have
*identical* band structures. Every observable built from band energies —
total energy, gap, density of states — is blind to the difference. Yet one
pattern, cut open, produces states at exactly zero energy bound to its ends,
and the other produces none. Whatever distinguishes them is not in the
spectrum; it is in the *eigenstates* — specifically, in how the Bloch state
$|u_k\rangle$ twists as $k$ crosses the Brillouin zone. That twist is a
Berry phase {cite}`berry1984`, its crystal version is Zak's phase
{cite}`zak1989`, and this notebook builds the machinery from first
principles and lets it loose on the simplest model that has any: the
Su–Schrieffer–Heeger (SSH) chain {cite}`su1979`, written down for
polyacetylene and now the hydrogen atom of topological matter
{cite}`asboth2016`.

The build: a *discrete* Berry phase — the phase of a product of overlaps,
gauge-invariant by construction — certified on Berry's original example (a
spin-$\tfrac12$ dragged around a cone picks up half the enclosed solid
angle, and the computation reproduces $\gamma = -m\Omega$ to $10^{-5}$ with
the predicted $O(1/N^2)$ convergence). Then the SSH chain: its $2\times2$
Bloch Hamiltonian, the winding of $\mathbf d(k)$ that cleanly splits the two
dimerizations, the Zak phase quantized to $0$ or $\pi$ at machine precision
and *provably* gauge-independent (we re-run it through deliberately
randomized eigenvector phases). The payoffs follow, each computed: edge
states at $|E| \sim 10^{-10}$ living on one sublattice to $10^{-31}$, with
localization length exactly $1/[2\ln(w/v)]$; Wannier functions from an
`ifft`, exponentially localized in a smooth gauge and destroyed by a random
one (spreads 0.08 versus 300 — the Marzari–Vanderbilt lesson
{cite}`marzari1997`); and the modern theory of polarization
{cite}`kingsmith1993` — the Wannier center *is* the polarization, two
independent formulas agree, and $P$ jumps by exactly $e/2$ across the
transition. The verdict exercise builds a chain with winding number *two*
and finds its promised *four* zero modes: bulk–boundary correspondence,
counted.

> **Conventions (this notebook).** Spinless electrons; the lower band
> filled. The SSH cell holds sites $A$ and $B$; $v \geq 0$ is the
> *intracell* hopping $A\!-\!B$ and $w \geq 0$ the *intercell* hopping
> $B\!-\!A$; lattice constant $a = 1$ and hoppings in units of $w$ unless
> stated. The Bloch Hamiltonian uses the cell-periodic convention
> $h(k) = v + we^{-ik}$ (Eq. {eq}`eq-bws-ssh`), so $|u_k\rangle$ does not
> carry intracell positions; where physical positions matter (Resta's
> formula) we say so. Eigenvectors come from `numpy.linalg.eigh`
> (lower band = column 0), Berry phases from `numpy.angle` of an overlap
> product, Wannier functions from `numpy.fft.ifft`, and randomness from
> `numpy.random.default_rng(1966)`.
>
> **How to read the checks.** Each exercise closes with a `validate` call
> against an independent fact: an exact quantization, a closed-form
> localization length, a gauge-invariance rerun, a counted number of zero
> modes. A ✓ is strong evidence; a ✗ is a prompt to locate the
> discrepancy, not an automatic verdict.
>
> **Scope.** One dimension, one filled band, no interactions: Zak phase,
> winding number, Wannier functions, polarization. The two-dimensional
> story (Chern numbers, quantum Hall, $\mathbb{Z}_2$ insulators) needs
> Berry *curvature* and is surveyed in Asbóth–Oroszlány–Pályi
> {cite}`asboth2016` and Vanderbilt {cite}`vanderbilt2018`, whose Ch. 3–4
> this notebook follows in spirit; interactions wait for
> [§8.13](hubbard-model.ipynb).

## Theory in brief

### Berry's phase, discretized

Drag a Hamiltonian $\hat H(\boldsymbol\lambda)$ slowly around a closed loop
in parameter space and the adiabatic theorem
([§6.24](../06-quantum-mechanics/time-dependent-perturbation.ipynb)) keeps
the system in the instantaneous eigenstate — but the state returns with a
phase beyond the dynamical $e^{-i\int E\,dt/\hbar}$. Berry's discovery
{cite}`berry1984` is that the extra phase is *geometric*: it depends only on
the loop, not the speed. On a computer the loop is $N$ discrete points
$|u_1\rangle, |u_2\rangle, \dots, |u_N\rangle, |u_1\rangle$, and the
geometric phase is the phase of the product of neighboring overlaps,

```{math}
:label: eq-bws-berry
\gamma \;=\; -\operatorname{Im}\,\ln \prod_{j=1}^{N}
\langle u_j | u_{j+1} \rangle ,
\qquad |u_{N+1}\rangle \equiv |u_1\rangle .
```

This formula is the workhorse of the whole subject, and its crucial
property is visible by inspection: multiply any $|u_j\rangle$ by an
arbitrary phase $e^{i\alpha_j}$ (a *gauge transformation* — eigensolvers do
this to us constantly) and the phase enters one overlap as $e^{-i\alpha_j}$
(bra) and the adjacent one as $e^{+i\alpha_j}$ (ket): the product is
untouched. Gauge invariance is not a theorem to invoke; it is a
cancellation you can point at — and Exercise 3 tests it with a random
number generator. For a spin-$\tfrac12$ in a field along
$\hat{\mathbf n}$, transported around a circle of colatitude $\theta$, the
answer is Berry's celebrated

```{math}
:label: eq-bws-spin
\gamma_m \;=\; -\,m\,\Omega, \qquad \Omega = 2\pi(1 - \cos\theta) :
```

each eigenstate ($m = \pm\tfrac12$) acquires minus $m$ times the solid
angle the loop subtends — geometry, with no dynamics anywhere in it.

### The SSH chain and its winding number

The dimerized chain of [§8.9](tight-binding.ipynb)'s Peierls exercise,
relabeled: two sites per cell, intracell hopping $v$, intercell hopping
$w$. In the Bloch basis,

```{math}
:label: eq-bws-ssh
H(k) = \begin{pmatrix} 0 & h(k) \\ h^*(k) & 0 \end{pmatrix},
\qquad h(k) = v + w\,e^{-ik},
\qquad \varepsilon_\pm(k) = \pm\,|h(k)| ,
```

with gap $2|v - w|$, closing only at $v = w$ (at $k = \pi$) — the undimerized
metallic chain. Because the Hamiltonian has no diagonal terms, it
*anticommutes* with $\sigma_z$: chiral symmetry, which pins the spectrum
symmetric about zero and will pin edge states *at* zero. Write
$H(k) = \mathbf d(k)\cdot\boldsymbol\sigma$ with
$\mathbf d = (v + w\cos k,\; w\sin k,\; 0)$: as $k$ crosses the zone,
$\mathbf d(k)$ traces a circle of radius $w$ centered at $(v, 0)$. The
circle either encloses the origin ($w > v$) or does not ($v > w$), and no
smooth deformation that keeps the gap open ($\mathbf d \neq 0$) can change
which — an integer has been hiding in the Hamiltonian:

```{math}
:label: eq-bws-winding
W \;=\; \frac{1}{2\pi} \oint d\theta_d
\;=\; \frac{1}{2\pi} \int_{-\pi}^{\pi}
\frac{d}{dk}\big[\arg\!\big(d_x(k) + i\,d_y(k)\big)\big]\, dk \;\in\; \mathbb{Z} ,
```

counted with the orientation of the $\mathbf d$-loop. Since
$d_x + i\,d_y = h^*(k) = v + w\,e^{+ik}$, this is the winding of $h^*$,
the form the code (and Exercise 7's extended chain) uses; winding
$\arg h(k)$ itself would flip the sign to $W = -1$ in the topological
phase.

### Zak phase, Wannier functions, and polarization

Berry's loop specialized to the Brillouin zone (a closed loop, since
$k = -\pi$ and $\pi$ are the same point) is the **Zak phase**
{cite}`zak1989` of a band. Inversion symmetry forces
$\gamma_{\mathrm{Zak}} \in \{0, \pi\}$ mod $2\pi$ — a two-valued label
protected as long as the gap stays open, and equal to $\pi W$ mod $2\pi$
here. Its physical meaning runs through **Wannier functions**, the Fourier
transforms of the Bloch states,

```{math}
:label: eq-bws-wannier
|w_R\rangle \;=\; \frac{1}{N}\sum_k e^{-ikR}\,|u_k\rangle ,
\qquad
\bar x \;=\; \langle w_0|\,\hat x\,|w_0\rangle
\;=\; \frac{\gamma_{\mathrm{Zak}}}{2\pi}\,a \;\;(\mathrm{mod}\; a) :
```

the Zak phase *is* the position of the band's charge within the cell. This
is the modern theory of polarization {cite}`kingsmith1993`: $P = e\bar x/a$
per cell, defined only modulo $e$ (choosing which cell is "home" shifts
$\bar x$ by $a$), so absolute $P$ is convention — but *changes* in $P$ are
physical, measurable charge flow. The catch in Eq. {eq}`eq-bws-wannier` is
the gauge: the $|u_k\rangle$ must vary *smoothly* with $k$, or the Fourier
sum shreds the Wannier function across the whole crystal. Exercise 5
measures the damage, and Marzari–Vanderbilt {cite}`marzari1997` built the
modern localized-orbital industry on repairing it.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from ecp import validate

INK, AMBER, SOFT = "#16213e", "#c0851a", "#46506b"

RNG = np.random.default_rng(1966)


def berry_phase(states):
    """Discrete Berry phase of a closed loop of normalized states.

    Implements Eq. (eq-bws-berry): the negative phase of the product of
    neighboring overlaps around the loop, closing it with <u_N|u_1>.
    Gauge-invariant by construction: per-state phases cancel between
    bra and ket.

    Parameters
    ----------
    states : numpy.ndarray
        Array of shape (n_points, dim); row j is the state at loop point j
        (the closing overlap back to row 0 is added internally).

    Returns
    -------
    float
        The Berry phase in [-pi, pi).
    """
    prod = 1.0 + 0.0j
    n_pts = len(states)
    for j in range(n_pts):
        prod *= np.vdot(states[j], states[(j + 1) % n_pts])
    return float(-np.angle(prod))


def ssh_bloch(k, v, w):
    """SSH Bloch Hamiltonian, Eq. (eq-bws-ssh), in the cell-periodic gauge.

    Parameters
    ----------
    k : float
        Crystal momentum (lattice constant a = 1).
    v, w : float
        Intracell and intercell hopping amplitudes.

    Returns
    -------
    numpy.ndarray
        The (2, 2) complex Hermitian Bloch matrix.
    """
    off = v + w * np.exp(-1j * k)
    return np.array([[0.0, off], [np.conj(off), 0.0]])


def lower_band(v, w, n_k, randomize=None):
    """Lower-band Bloch eigenvectors on a uniform k-grid over the zone.

    Parameters
    ----------
    v, w : float
        SSH hoppings.
    n_k : int
        Number of k-points, k_j = 2 pi j / n_k.
    randomize : numpy.random.Generator, optional
        If given, multiply every eigenvector by a random phase — a
        deliberately pathological gauge for invariance tests.

    Returns
    -------
    numpy.ndarray
        Array of shape (n_k, 2): row j is u(k_j) for the lower band.
    """
    states = np.zeros((n_k, 2), dtype=complex)
    for j in range(n_k):
        k = 2.0 * np.pi * j / n_k
        _, vecs = np.linalg.eigh(ssh_bloch(k, v, w))
        u = vecs[:, 0]
        if randomize is not None:
            u = u * np.exp(2j * np.pi * randomize.random())
        states[j] = u
    return states


def ssh_open(n_cells, v, w, w2=0.0):
    """Open SSH chain Hamiltonian (2 n_cells sites), optionally extended.

    Sites alternate A, B, A, B, ...; bond A_i-B_i carries v, B_i-A_{i+1}
    carries w, and B_i-A_{i+2} carries the second-neighbor intercell w2
    (zero for the standard model). All couplings connect A to B only, so
    chiral symmetry is exact for every (v, w, w2).

    Parameters
    ----------
    n_cells : int
        Number of unit cells (open boundaries).
    v, w : float
        Intracell and intercell hoppings.
    w2 : float, optional
        Long-range intercell hopping B_i-A_{i+2} (default 0).

    Returns
    -------
    numpy.ndarray
        The (2 n_cells, 2 n_cells) real symmetric Hamiltonian.
    """
    n = 2 * n_cells
    ham = np.zeros((n, n))
    for i in range(n_cells):
        ham[2 * i, 2 * i + 1] = ham[2 * i + 1, 2 * i] = v
        if i < n_cells - 1:
            ham[2 * i + 1, 2 * i + 2] = ham[2 * i + 2, 2 * i + 1] = w
        if w2 != 0.0 and i < n_cells - 2:
            ham[2 * i + 1, 2 * i + 4] = ham[2 * i + 4, 2 * i + 1] = w2
    return ham


def winding_number(v, w, w2=0.0, n_k=2000):
    """Winding number of h(k) = v + w e^{ik} + w2 e^{2ik} around the origin.

    Discretizes Eq. (eq-bws-winding): unwrap the phase of h along the zone
    and count full turns. (The orientation convention e^{+ik} makes W
    positive in the topological phase.)

    Parameters
    ----------
    v, w : float
        SSH hoppings.
    w2 : float, optional
        Second-neighbor intercell hopping (default 0).
    n_k : int, optional
        Number of k-points for the discretized loop.

    Returns
    -------
    float
        The winding number (an integer up to discretization noise).
    """
    ks = 2.0 * np.pi * np.arange(n_k + 1) / n_k
    h_of_k = v + w * np.exp(1j * ks) + w2 * np.exp(2j * ks)
    return float(np.diff(np.unwrap(np.angle(h_of_k))).sum() / (2.0 * np.pi))

## Exercise 1 — Berry's phase, certified on Berry's example

Before trusting Eq. {eq}`eq-bws-berry` on a crystal, certify it on the
problem Berry solved by hand: a spin-$\tfrac12$ in a unit field
$\hat H = \hat{\mathbf n}\cdot\boldsymbol\sigma$, with $\hat{\mathbf n}$
dragged once around a circle of colatitude $\theta$.

**Part a)** For $N = 400$ points on the loop, build the lower eigenstate
($m = -\tfrac12$, column 0 of `numpy.linalg.eigh`) at each point and
evaluate the Berry phase with `berry_phase`. Sweep 21 colatitudes
$\theta \in [0.15, \pi - 0.15]$ and compare with Eq. {eq}`eq-bws-spin`,
$\gamma = +\Omega/2 = \pi(1 - \cos\theta)$, wrapped to $(-\pi, \pi]$ —
phases are defined mod $2\pi$, so the theory curve *jumps* branch at
$\theta = \pi/2$, and the computation should follow it. Compare phase
distances $|e^{i\gamma} - e^{i\Omega/2}|$, not raw differences.

**Part b)** Discretization converges quadratically: at $\theta = \pi/3$,
compute the error at $N = 100$ and $N = 1000$ and confirm the ratio is
$\approx 100$ — the $O(1/N^2)$ signature of the overlap-product formula.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1 — geometry against Berry's closed form

The sweep must match $\gamma = \Omega/2$ in phase distance at the $10^{-4}$
level for $N = 400$, and refining $100 \to 1000$ points must shrink the
error a hundredfold.

In [ ]:
validate.check(
    bool(spin_err < 1e-4),
    "gamma = Omega/2 across 21 colatitudes (N = 400)",
    f"max phase distance {spin_err:.2e}",
)
validate.close(
    conv_ratio, 100.0, "O(1/N^2) convergence of the discrete loop", rtol=5e-2
)

## Exercise 2 — The SSH chain: one spectrum, two insulators

Now the crystal. The machinery meets the model whose entire point is that
the *spectrum* cannot tell its two phases apart.

**Part a)** Plot $\varepsilon_\pm(k) = \pm|v + we^{-ik}|$ over the zone for
$(v, w) = (1, 0.5)$, $(1, 1)$, and $(0.5, 1)$. Report the gaps and confirm
the closed form $2|v - w|$: the first and third cases are *identical*
spectra with gap $1$, and the middle is the metal where the two insulators
meet.

**Part b)** Plot the loop $\mathbf d(k) = (v + w\cos k,\; w\sin k)$ for the
two insulators — the emblem of the subject. Compute the winding number by
the discretized Eq. {eq}`eq-bws-winding` for both, and check
$W = 0$ (trivial, origin outside) versus $W = 1$ (topological, origin
enclosed) to $10^{-12}$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 2 — gaps and windings

Both insulators must show gap exactly $2|v - w| = 1$, the metal must close
it, and the discretized winding integral must return the exact integers.

In [ ]:
gap_triv = 2.0 * np.abs(1.0 + 0.5 * np.exp(-1j * k_arr)).min()
gap_topo = 2.0 * np.abs(0.5 + 1.0 * np.exp(-1j * k_arr)).min()
gap_metal = 2.0 * np.abs(1.0 + 1.0 * np.exp(-1j * k_arr)).min()
validate.close(gap_triv, 1.0, "trivial gap = 2|v - w|", rtol=1e-9)
validate.close(gap_topo, 1.0, "topological gap = 2|v - w|", rtol=1e-9)
validate.check(
    bool(gap_metal < 1e-2), "the v = w metal closes the gap", f"gap {gap_metal:.1e}"
)
validate.close(wind_triv, 0.0, "winding number, trivial phase", atol=1e-12)
validate.close(wind_topo, 1.0, "winding number, topological phase", atol=1e-12)

## Exercise 3 — The Zak phase, quantized and gauge-proof

The Berry machine, aimed down the Brillouin zone.

**Part a)** With `lower_band` ($n_k = 200$) and `berry_phase`, compute the
Zak phase for $(v, w) = (1, 0.5)$, $(1, 0.8)$, $(0.8, 1)$, and $(0.5, 1)$.
It must quantize: $0$ whenever $v > w$, $\pm\pi$ (the same point on the
phase circle) whenever $w > v$ — and not approximately, but to better than
$10^{-10}$, because inversion symmetry forces the product in Eq.
{eq}`eq-bws-berry` to be real. Compare $e^{i\gamma}$ against $\pm 1$, the
quantization's gauge-safe form.

**Part b)** Prove the gauge invariance experimentally: rerun both phases
with `randomize=RNG`, which multiplies every eigenvector by an independent
random phase — the worst gauge a malicious eigensolver could produce — and
confirm $e^{i\gamma}$ is unchanged at $10^{-8}$.

**Part c)** Convert to Wannier centers by Eq. {eq}`eq-bws-wannier`:
$\bar x = (\gamma/2\pi) \bmod 1$, which must land on exactly $0$ (trivial)
and exactly $\tfrac12$ (topological) — the band's charge sits *on* the cell
in one phase and *between* cells in the other.

In [ ]:
# (solution hidden on the public site)


### Validation 3 — quantized, and provably gauge-free

Quantization to $\{0, \pi\}$ at $10^{-10}$, gauge invariance under full
phase randomization at $10^{-8}$, and Wannier centers on $0$ and $a/2$.

In [ ]:
validate.check(
    bool(quant_err < 1e-10),
    "Zak phase quantized to {0, pi} in all four cases",
    f"max deviation {quant_err:.1e}",
)
validate.check(
    bool(gauge_err < 1e-8),
    "Zak phase survives a fully randomized gauge",
    f"max deviation {gauge_err:.1e}",
)
validate.close(xbar_triv, 0.0, "trivial Wannier center at 0", atol=1e-10)
validate.close(xbar_topo, 0.5, "topological Wannier center at a/2", atol=1e-10)

## Exercise 4 — Bulk–boundary correspondence: the edge states appear

The bulk invariant now makes a claim about *surfaces*: cut the chain open,
and zero-energy end states must exist exactly when $W = 1$. This is the
bulk–boundary correspondence, and it is checkable to ten digits.

**Part a)** Diagonalize the open chain (`ssh_open`, $N = 40$ cells) with
`numpy.linalg.eigh` for $v/w$ swept across the transition ($w = 1$), and
plot the spectral flow: two states must detach from the bands and collapse
onto $E = 0$ precisely as $v/w$ drops below $1$.

**Part b)** Pin the numbers at $(v, w) = (0.5, 1)$: for $N = 30$ the
midgap pair must sit at $|E| < 10^{-8}$ (measured: $7.0\times10^{-10}$),
while the trivial chain's smallest $|E|$ stays above $0.4$. The splitting
is the two ends' exponentially small overlap, so growing $N = 20 \to 30$
must shrink it by exactly $(w/v)^{10} = 1024$ — a scaling law with no
fitted constants.

**Part c)** Chiral symmetry makes a sharper claim: a zero mode is an
eigenstate of $\sigma_z$, so the left edge state lives *entirely* on the
$A$ sublattice. Form the left-localized combination of the (numerically
hybridized) midgap pair and measure its $B$-sublattice weight — it is not
small, it is $10^{-31}$.

**Part d)** The decay is geometric, $|\psi_A(m)|^2 \propto (v/w)^{2m}$:
fit the log-slope of the cell weights and confirm the localization length
$\xi = 1/[2\ln(w/v)]$ at $v = 0.3, 0.5, 0.7, 0.8$ — four couplings, four
ratios of $1.000$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 4 — the boundary keeps the bulk's promise

Zero modes present at $10^{-8}$ where $W = 1$ and absent where $W = 0$;
the parameter-free $(w/v)^{10}$ splitting law; sublattice purity; and the
closed-form localization length at four couplings.

In [ ]:
validate.check(
    bool(abs(e30[0]) < 1e-8),
    "edge pair at zero energy in the topological phase",
    f"|E| = {abs(e30[0]):.1e}",
)
validate.check(
    bool(e_trivial > 0.4),
    "no midgap states in the trivial phase",
    f"min |E| = {e_trivial:.3f}",
)
validate.close(
    split_ratio, 1024.0, "splitting shrinks by (w/v)^10 for N = 20 -> 30", rtol=1e-2
)
validate.check(
    bool(weight_b < 1e-20),
    "left edge state pure A-sublattice (chiral symmetry)",
    f"B weight {weight_b:.1e}",
)
validate.close(
    float(np.max(np.abs(np.array(xi_ratios) - 1.0))),
    0.0,
    "localization length = 1/[2 ln(w/v)] at four couplings",
    atol=1e-2,
)

## Exercise 5 — Wannier functions: localization is a gauge you must earn

Equation {eq}`eq-bws-wannier` looks innocent — an inverse Fourier
transform — but it silently assumes the $|u_k\rangle$ vary smoothly with
$k$. Eigensolvers promise no such thing.

**Part a)** Build the lower-band Wannier function on a ring of 64 cells by
`numpy.fft.ifft` of the Bloch states in a *smooth* gauge (fix each
$u_k$'s first component real positive) for both phases, and in the
random gauge for comparison. Plot the cell weights on a log scale.

**Part b)** Quantify with the spread
$\Omega = \langle m^2\rangle - \langle m\rangle^2$ over cell index: the
smooth gauge gives $0.083$ (trivial) and $0.333$ (topological) cell²,
the random gauge $\sim 300$ — the state shredded over the whole ring.
This factor of $10^3$ is why the Marzari–Vanderbilt functional
{cite}`marzari1997` (minimize $\Omega$ over gauges) exists.

**Part c)** The smooth-gauge tails are exponential with the rate set by
the gap: fit the log-slope of the tail at $v/w = 1.43$, $2$, and $3.33$
and compare with $2\ln(v/w)$ — the same analytic structure (the branch
point of $|h(k)|$ continued to complex $k$) that set the edge state's
$\xi$. The finite-window fit runs $5$–$18\%$ steep because the tail
carries a power-law prefactor on top of the exponential; the *ordering*
and the rates land within $20\%$, and honesty about that window effect is
part of the exercise.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5 — localized, shredded, and gap-rated

Smooth-gauge spreads at their measured values, the random gauge worse by
two orders of magnitude, and tail rates tracking $2\ln(v/w)$ within the
window-honest $20\%$ — and *monotonically* in the gap.

In [ ]:
validate.close(s_triv, 0.0833, "smooth trivial Wannier spread", rtol=5e-2)
validate.close(s_topo, 0.3333, "smooth topological Wannier spread", rtol=5e-2)
validate.check(
    bool(s_rand > 100.0 * s_topo),
    "random gauge shreds the Wannier function",
    f"spread {s_rand:.0f} vs {s_topo:.3f} cell^2",
)
rate_targets = [2.0 * np.log(1.0 / 0.7), 2.0 * np.log(2.0), 2.0 * np.log(1.0 / 0.3)]
validate.check(
    bool(
        all(abs(r / t - 1.0) < 0.2 for r, t in zip(tail_rates, rate_targets))
        and tail_rates[0] < tail_rates[1] < tail_rates[2]
    ),
    "Wannier tails exponential at the gap's rate (within window's 20%)",
    f"rates {tail_rates[0]:.2f} < {tail_rates[1]:.2f} < {tail_rates[2]:.2f}",
)

## Exercise 6 — Polarization: only the jump is physical

The Wannier center of Exercise 3 is, by the modern theory
{cite}`kingsmith1993`, the electronic polarization: $P = e\bar x/a$ per
cell, *defined only modulo* $e$. This exercise computes $P$ by two
independent routes and shows that where the routes disagree is exactly
where polarization was never well-defined to begin with — while the *jump*
at the transition is route-independent and exactly $e/2$.

**Part a)** Scan the Zak-phase Wannier center $\bar x(v)$ across
$v/w \in [0.5, 1.5]$: a step function, $\tfrac12$ then $0$, switching at
the gap closing — no intermediate values, because inversion symmetry
allows none. Confirm the jump is $\tfrac12$ even between $v/w = 0.99$ and
$1.01$.

**Part b)** Recompute $P$ by Resta's many-body formula {cite}`resta1998`,
built with *physical* site positions ($B$ at $+a/2$ within the cell) on a
closed ring of 64 cells with the lower band filled:
$P = \frac{1}{2\pi}\operatorname{Im}\ln\det S$ with
$S_{mn} = \langle\psi_m|e^{2\pi i\hat x/L}|\psi_n\rangle$. It returns
$0.75$ and $0.25$ — a quarter cell *below* the Zak plateaus (mod $a$,
the same shift on both). Two ingredients add up: the bonding charge
sits mid-bond, an intracell offset of $+\tfrac14$ (dimer-limit bond
centers at $m + \tfrac14$ trivial, $m + \tfrac34$ topological) — and
Resta's phase reports the *ring-averaged* center
$\sum_m (m + x_{\mathrm{phys}})/N \bmod 1$, whose mean cell index
$(N-1)/2$ contributes an extra $\tfrac12$ for the even ring $N = 64$
(an odd ring returns $0.25$ and $0.75$ — try it). Neither absolute
number is wrong; absolute
$P$ is convention. The difference $P_{\mathrm{triv}} - P_{\mathrm{topo}}$
must be exactly $\tfrac12$ by *both* routes — that is the physical,
quantized $e/2$: the fractional charge SSH attached to polyacetylene's
domain walls {cite}`su1979`.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6 — e/2, by two unrelated formulas

Both routes must produce a jump of exactly $\tfrac12$, their mutual offset
must be the same $-\tfrac14$ (mod 1) shift on both plateaus — the $+\tfrac14$
basis offset plus the even ring's half cell, $\tfrac14 + \tfrac12 \equiv
-\tfrac14$ — and the step must be complete within $1\%$ of the transition.

In [ ]:
validate.close(jump_zak, 0.5, "polarization jump e/2 (Zak route)", atol=1e-6)
validate.close(jump_resta, 0.5, "polarization jump e/2 (Resta route)", atol=1e-3)
off_triv = (resta_center(1.5, 1.0) - zak_center(1.5, 1.0) + 0.25 + 0.5) % 1.0 - 0.5
off_topo = (resta_center(0.5, 1.0) - zak_center(0.5, 1.0) + 0.25 + 0.5) % 1.0 - 0.5
validate.close(
    off_triv,
    0.0,
    "offset -1/4 mod 1 (basis + even-ring half cell), trivial side",
    atol=1e-3,
)
validate.close(
    off_topo,
    0.0,
    "offset -1/4 mod 1 (basis + even-ring half cell), topological side",
    atol=1e-3,
)

## Exercise 7 — Verdict: winding number two, and the count that proves it

If $W$ really is an integer — not a binary — there should be chains with
$W = 2$, and the correspondence should promise *four* zero modes. Add a
second-neighbor intercell hop $w_2$ ($B_i \to A_{i+2}$, still $A$–$B$ only,
so chiral symmetry survives): $h^*(k) = v + we^{ik} + w_2 e^{2ik}$ can now
wind twice.

**Part a)** Compute $W$ for the three regimes — $v$-dominant $(1, 0.3,
0.2)$, $w$-dominant $(0.3, 1, 0.2)$, $w_2$-dominant $(0.2, 0.3, 1)$ — and
plot the three $h(k)$ loops: zero, one, and two turns around the origin.

**Part b)** Diagonalize the open chains ($N = 40$ cells) and *count* the
zero modes ($|E| < 10^{-6}$): the counts must be $0$, $2$, and $4$ — two
per edge in the doubly wound phase, at $|E| \sim 10^{-14}$, with the rest
of the spectrum gapped at $0.756$. Print the verdict table for the whole
notebook.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7 — the integer counts states

Windings $0$, $1$, $2$ exact; zero-mode counts $0$, $2$, $4$ exact; and
the doubly wound modes at machine zero with a genuine gap above them.

In [ ]:
validate.close(windings["v-dominant"], 0.0, "extended chain: W = 0", atol=1e-10)
validate.close(windings["w-dominant"], 1.0, "extended chain: W = 1", atol=1e-10)
validate.close(windings["w2-dominant"], 2.0, "extended chain: W = 2", atol=1e-10)
validate.check(
    bool(
        zero_counts["v-dominant"] == 0
        and zero_counts["w-dominant"] == 2
        and zero_counts["w2-dominant"] == 4
    ),
    "zero-mode counts = 2|W| at both edges",
    f"counts {[zero_counts[n] for n, *_ in regimes]}",
)
validate.check(
    bool(spectra["w2-dominant"][3] < 1e-12 and spectra["w2-dominant"][4] > 0.7),
    "four modes at machine zero, gapped from the bands",
    f"|E|_4 = {spectra['w2-dominant'][3]:.1e}, |E|_5 = {spectra['w2-dominant'][4]:.3f}",
)

:::{admonition} With your assistant
:class: tip
The e/2 jump hints at something bigger: make $v$ and $w$ *time-dependent*
and the polarization can be pumped. The Rice–Mele cycle adds a staggered
on-site term $\pm u$ to Eq. {eq}`eq-bws-ssh` and steers
$(v - w, u)$ once around the origin — around the gap closing. Have your
assistant build the cycle ($v - w = \cos\tau$, $u = \sin\tau$, say, with
$w = 1$) and track the Wannier center $\bar x(\tau)$ through one period
with the Zak machinery above. Then run the check that is yours alone: the
accumulated center must wind *exactly once* — total displacement
$\Delta\bar x = 1.0$ cell per cycle (`numpy.unwrap` on $2\pi\bar x$,
`numpy.isclose` with `atol=1e-6`) — Thouless's quantized pump, the Chern
number smuggled into one dimension plus time. The check is yours.
:::

## Notebook summary

The volume's window on topology opened with a six-line formula. The
discrete Berry phase — `numpy.angle` of an overlap product — reproduced
Berry's solid-angle law across 21 loop geometries at $2.5\times10^{-5}$
with the predicted hundredfold error drop from $N = 100$ to $1000$, and
its gauge invariance was demonstrated rather than asserted: full phase
randomization of every eigenvector moved the Zak phase by less than
$10^{-14}$. Aimed at the SSH chain, the machinery split what the spectrum
cannot: two dimerizations with identical bands carry windings $0$ and $1$,
Zak phases $0$ and $\pi$ quantized to eleven digits, and Wannier centers
at $0$ and $a/2$. The bulk's promises were kept at the boundary — edge
states at $7\times10^{-10}$ obeying the parameter-free $(w/v)^{10}$
splitting law (measured $1024.0$ against $1024$), pure $A$-sublattice to
$10^{-31}$, localization length $1/[2\ln(w/v)]$ at four couplings within
$1\%$ — and in the charge: smooth-gauge Wannier functions localized to
$0.08$–$0.33$ cell² (a random gauge shreds them to $300$), polarization
computed by two unrelated formulas that disagree about the convention and
agree that the jump is exactly $e/2$. The verdict chain wound twice and
produced exactly four zero modes at $10^{-14}$: the integer counts states.

## Outlook

- Everything here was one filled band in one dimension. In two dimensions
  the Berry phase per unit area — the Berry *curvature* — integrates to
  the Chern number, and its boundary promise is the quantum Hall effect's
  dissipationless edge currents; Vanderbilt {cite}`vanderbilt2018` carries
  exactly this notebook's machinery there.
- The Wannier functions built by `ifft` become, in production codes, the
  maximally localized orbitals of Marzari–Vanderbilt {cite}`marzari1997` —
  the standard post-processing of the DFT bands of
  [§8.8](exact-conditions-band-gap.ipynb) and the natural basis for the
  models of the next movement.
- A localized orbital with a known center is precisely where interactions
  bite hardest. Put a repulsion $U$ on it and the band picture faces its
  reckoning: the Hubbard model of [§8.13](hubbard-model.ipynb), where
  Movement IV begins.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()